# Time Series Forecasting Models
## Task 2: Initial Forecasting Model Implementation

This notebook implements ARIMA models for forecasting stock prices.

### Models Implemented:
1. ARIMA (AutoRegressive Integrated Moving Average)
2. SARIMA (Seasonal ARIMA)
3. LSTM Neural Network

**Dataset**: TSLA, BND, SPY prices (2015-01-01 to 2026-06-30)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import MinMaxScaler

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## 1. Load Data and Prepare for Modeling

In [ ]:
# Load processed data
prices = pd.read_csv('../data/raw/stock_prices.csv', index_col=0, parse_dates=True)
returns = pd.read_csv('../data/processed/daily_returns.csv', index_col=0, parse_dates=True)

print("Data loaded successfully!")
print(f"Price data shape: {prices.shape}")
print(f"Returns data shape: {returns.shape}")

# Select ticker for modeling
TICKER = 'SPY'  # Can change to TSLA or BND
target_data = prices[TICKER]
target_returns = returns[TICKER].dropna()

print(f"\nModeling: {TICKER}")
print(f"Date range: {target_data.index.min()} to {target_data.index.max()}")
display(target_data.head())

## 2. Chronological Train/Test Split
### Splitting by date (not random shuffle) to preserve temporal order

In [ ]:
# Chronological Train/Test Split
# Use last 20% of data for testing
TEST_SIZE = 0.20

split_idx = int(len(target_data) * (1 - TEST_SIZE))

train_prices = target_data.iloc[:split_idx]
test_prices = target_data.iloc[split_idx:]

train_returns = target_returns.iloc[:split_idx]
test_returns = target_returns.iloc[split_idx:]

print("=" * 60)
print("TRAIN/TEST SPLIT")
print("=" * 60)
print(f"\nTotal observations: {len(target_data)}")
print(f"Training set: {len(train_prices)} observations")
print(f"  Date range: {train_prices.index.min()} to {train_prices.index.max()}")
print(f"Test set: {len(test_prices)} observations")
print(f"  Date range: {test_prices.index.min()} to {test_prices.index.max()}")
print(f"\nSplit ratio: {TEST_SIZE*100}% test, {(1-TEST_SIZE)*100}% train")

In [ ]:
# Visualization of Train/Test Split
fig, ax = plt.subplots(figsize=(14, 6))

train_prices.plot(ax=ax, label='Training Data', color='blue', linewidth=1.5)
test_prices.plot(ax=ax, label='Test Data', color='red', linewidth=1.5)

ax.axvline(x=test_prices.index[0], color='black', linestyle='--', linewidth=2, label='Split Point')
ax.set_title(f'{TICKER} Train/Test Split (Time-Based)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Close Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/train_test_split.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/train_test_split.png")

## 3. ARIMA Model Analysis

### 3.1 Check Stationarity and Determine Differencing Order (d)

In [ ]:
# Stationarity analysis for differencing order
def check_stationarity(series, max_diff=3):
    """Check stationarity at different differencing levels."""
    results = []
    
    for d in range(max_diff + 1):
        if d == 0:
            diff_series = series
        else:
            diff_series = series.diff(d).dropna()
        
        adf_result = adfuller(diff_series.dropna(), autolag='AIC')
        
        results.append({
            'd': d,
            'ADF Statistic': adf_result[0],
            'p-value': adf_result[1],
            'Is Stationary': adf_result[1] < 0.05
        })
    
    return pd.DataFrame(results)

# Check stationarity for price data
print("### Stationarity Analysis for Price Data ###")
stationarity_df = check_stationarity(train_prices)
display(stationarity_df)

# Select differencing order
DIFF_ORDER = stationarity_df[stationarity_df['Is Stationary']]['d'].min()
print(f"\nSelected differencing order (d): {DIFF_ORDER}")

### 3.2 ACF/PACF Analysis for AR (p) and MA (q) Order Selection

In [ ]:
# Prepare differenced series for ACF/PACF
if DIFF_ORDER > 0:
    diff_prices = train_prices.diff(DIFF_ORDER).dropna()
else:
    diff_prices = train_prices

# Plot ACF and PACF
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ACF plot
plot_acf(diff_prices, ax=axes[0], lags=40)
axes[0].set_title('Autocorrelation Function (ACF)', fontsize=12)
axes[0].set_xlabel('Lag')

# PACF plot
plot_pacf(diff_prices, ax=axes[1], lags=40, method='ywm')
axes[1].set_title('Partial Autocorrelation Function (PACF)', fontsize=12)
axes[1].set_xlabel('Lag')

plt.tight_layout()
plt.savefig('../data/processed/acf_pacf.png', dpi=300, bbox_inches='tight')
plt.show()

print("""
### ACF/PACF Interpretation Guide:
- ACF slow decay + PACF sharp cutoff after p lags → AR(p) model
- PACF slow decay + ACF sharp cutoff after q lags → MA(q) model
- Both decay slowly → ARMA(p, q) model

Looking at the plots:
- Significant PACF spikes suggest AR order (p)
- Significant ACF spikes suggest MA order (q)
""")

### 3.3 Fit ARIMA Model

In [ ]:
# Define ARIMA parameters
# Based on ACF/PACF analysis, we'll test several configurations
ARIMA_ORDERS = [
    (1, 1, 1),  # Simple model
    (2, 1, 2),  # Moderate complexity
    (1, 1, 0),  # AR only
    (0, 1, 1),  # MA only
]

print("### ARIMA Model Comparison ###")
arima_results = []

for order in ARIMA_ORDERS:
    try:
        model = ARIMA(train_prices, order=order)
        fitted_model = model.fit()
        
        print(f"ARIMA{order}: AIC={fitted_model.aic:.2f}, BIC={fitted_model.bic:.2f}")
        
        arima_results.append({
            'order': order,
            'AIC': fitted_model.aic,
            'BIC': fitted_model.bic,
            'model': fitted_model
        })
    except Exception as e:
        print(f"ARIMA{order}: Failed - {e}")

# Select best model by AIC
best_arima = min(arima_results, key=lambda x: x['AIC'])
print(f"\nBest ARIMA order: {best_arima['order']} (AIC: {best_arima['AIC']:.2f})")

# Save parameters
SELECTED_ORDER = best_arima['order']

In [ ]:
# Fit final ARIMA model with selected order
print(f"\n### Fitting ARIMA{SELECTED_ORDER} ###")

arima_model = ARIMA(train_prices, order=SELECTED_ORDER)
arima_fitted = arima_model.fit()

print(arima_fitted.summary())

In [ ]:
# Model Diagnostics
print("\n### Model Diagnostics ###")
print(f"AIC: {arima_fitted.aic:.2f}")
print(f"BIC: {arima_fitted.bic:.2f}")
print(f"Log-Likelihood: {arima_fitted.llf:.2f}")
print(f"HQIC: {arima_fitted.hqic:.2f}")

# Residual Analysis
residuals = arima_fitted.resid

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Residuals over time
axes[0].plot(residuals)
axes[0].set_title('Residuals Over Time')
axes[0].set_xlabel('Observation')
axes[0].set_ylabel('Residual')

# Residuals histogram
axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_title('Residuals Distribution')
axes[1].set_xlabel('Residual')

# ACF of residuals
plot_acf(residuals, ax=axes[2], lags=20)
axes[2].set_title('Residuals ACF')

plt.tight_layout()
plt.savefig('../data/processed/arima_diagnostics.png', dpi=300, bbox_inches='tight')
plt.show()
print("\nDiagnostics figure saved.")

## 4. Generate Forecasts for Test Period

In [ ]:
# Generate forecasts for test period
forecast_steps = len(test_prices)

print(f"### Generating {forecast_steps}-step Forecast ###")

# Get forecast
forecast_result = arima_fitted.get_forecast(steps=forecast_steps)
forecast = forecast_result.predicted_mean
forecast_conf = forecast_result.conf_int()

# Set proper index
forecast.index = test_prices.index
forecast_conf.index = test_prices.index

print(f"Forecast generated for {forecast.index[0]} to {forecast.index[-1]}")
print(f"\nForecast values (first 5):")
display(forecast.head())

In [ ]:
# Visualization: Actual vs Forecast
fig, ax = plt.subplots(figsize=(14, 6))

# Training data (last 100 points)
train_prices[-100:].plot(ax=ax, label='Training Data', color='blue', linewidth=1.5)

# Test data (actual)
test_prices.plot(ax=ax, label='Actual Test Data', color='green', linewidth=2)

# Forecast
forecast.plot(ax=ax, label='ARIMA Forecast', color='red', linewidth=2, linestyle='--')

# Confidence intervals
ax.fill_between(forecast_conf.index, 
                forecast_conf.iloc[:, 0], 
                forecast_conf.iloc[:, 1],
                color='red', alpha=0.2, label='95% CI')

ax.set_title(f'{TICKER} ARIMA{SELECTED_ORDER} Forecast vs Actual', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Close Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/arima_forecast.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/arima_forecast.png")

In [ ]:
# Calculate forecast metrics
def calculate_metrics(actual, predicted):
    """Calculate forecast evaluation metrics."""
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    mape = mean_absolute_percentage_error(actual, predicted) * 100
    mse = mean_squared_error(actual, predicted)
    
    # Mean Absolute Scaled Error (MASE)
    # Using naive forecast (random walk) as benchmark
    naive_mae = np.mean(np.abs(np.diff(actual)))
    mase = mae / naive_mae if naive_mae > 0 else np.inf
    
    return {
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'MSE': mse,
        'MASE': mase
    }

metrics = calculate_metrics(test_prices, forecast)

print("=" * 60)
print("ARIMA FORECAST EVALUATION METRICS")
print("=" * 60)
print(f"\nRMSE: ${metrics['RMSE']:.4f}")
print(f"MAE:  ${metrics['MAE']:.4f}")
print(f"MAPE: {metrics['MAPE']:.2f}%")
print(f"MASE: {metrics['MASE']:.4f}")

## 5. SARIMA Model (Seasonal ARIMA)

In [ ]:
# SARIMA with weekly seasonality (s=5 for daily financial data)
# SARIMA(p, d, q)(P, D, Q, s)

SEASONAL_ORDER = (1, 1, 1, 5)  # Weekly seasonality (5 trading days)
SARIMA_ORDER = (1, 1, 1)

print("### Fitting SARIMA(SARIMA_ORDER, SEASONAL_ORDER) ###")

try:
    sarima_model = SARIMAX(train_prices, 
                           order=SARIMA_ORDER,
                           seasonal_order=SEASONAL_ORDER,
                           enforce_stationarity=False,
                           enforce_invertibility=False)
    
    sarima_fitted = sarima_model.fit(disp=False)
    
    print(sarima_fitted.summary())
    
    # Generate SARIMA forecast
    sarima_forecast_result = sarima_fitted.get_forecast(steps=forecast_steps)
    sarima_forecast = sarima_forecast_result.predicted_mean
    sarima_forecast.index = test_prices.index
    
    print(f"\nSARIMA AIC: {sarima_fitted.aic:.2f}")
    print(f"ARIMA AIC: {arima_fitted.aic:.2f}")
    
except Exception as e:
    print(f"SARIMA fitting failed: {e}")
    sarima_forecast = None

In [ ]:
# Compare forecasts if SARIMA succeeded
if sarima_forecast is not None:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    test_prices.plot(ax=ax, label='Actual', color='green', linewidth=2)
    forecast.plot(ax=ax, label='ARIMA Forecast', color='red', linewidth=2, linestyle='--')
    sarima_forecast.plot(ax=ax, label='SARIMA Forecast', color='blue', linewidth=2, linestyle='-.')
    
    ax.set_title(f'{TICKER} Forecast Comparison', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Close Price ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../data/processed/forecast_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # SARIMA metrics
    sarima_metrics = calculate_metrics(test_prices, sarima_forecast)
    print(f"\nSARIMA Metrics:")
    print(f"RMSE: ${sarima_metrics['RMSE']:.4f}")
    print(f"MAPE: {sarima_metrics['MAPE']:.2f}%")

## 6. LSTM Neural Network Model

In [ ]:
# LSTM Model Implementation
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    
    LSTM_AVAILABLE = True
    print("TensorFlow/Keras available for LSTM model")
except ImportError:
    LSTM_AVAILABLE = False
    print("TensorFlow not available. Skipping LSTM model.")

In [ ]:
if LSTM_AVAILABLE:
    # Prepare data for LSTM (windowed sequences)
    WINDOW_SIZE = 60  # Use 60 days to predict next day
    
    # Normalize data
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train_prices.values.reshape(-1, 1))
    test_scaled = scaler.transform(test_prices.values.reshape(-1, 1))
    
    # Create sequences
    def create_sequences(data, window_size):
        X, y = [], []
        for i in range(window_size, len(data)):
            X.append(data[i-window_size:i])
            y.append(data[i])
        return np.array(X), np.array(y)
    
    X_train, y_train = create_sequences(train_scaled, WINDOW_SIZE)
    
    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"\nInput: {WINDOW_SIZE} days -> Output: 1 day forecast")

In [ ]:
if LSTM_AVAILABLE:
    # Build LSTM model
    print("### Building LSTM Model ###")
    
    model = Sequential([
        LSTM(units=64, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
        Dropout(0.2),
        LSTM(units=32, return_sequences=True),
        Dropout(0.2),
        LSTM(units=16, return_sequences=False),
        Dropout(0.2),
        Dense(units=1)
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    
    model.summary()

In [ ]:
if LSTM_AVAILABLE:
    # Train LSTM model
    print("\n### Training LSTM Model ###")
    
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    
    history = model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=32,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )
    
    # Plot training history
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title('Model Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    
    axes[1].plot(history.history['mae'], label='Train MAE')
    axes[1].plot(history.history['val_mae'], label='Val MAE')
    axes[1].set_title('Model MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('../data/processed/lstm_training.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
if LSTM_AVAILABLE:
    # Generate LSTM forecast for test period
    print("\n### Generating LSTM Forecast ###")
    
    # Use last WINDOW_SIZE days from training + test data for rolling prediction
    full_data = np.concatenate([train_scaled[-WINDOW_SIZE:], test_scaled])
    
    lstm_forecast = []
    
    for i in range(len(test_prices)):
        X = full_data[i:i+WINDOW_SIZE].reshape(1, WINDOW_SIZE, 1)
        pred = model.predict(X, verbose=0)[0, 0]
        lstm_forecast.append(pred)
    
    # Inverse transform
    lstm_forecast = scaler.inverse_transform(np.array(lstm_forecast).reshape(-1, 1)).flatten()
    
    # Calculate LSTM metrics
    lstm_metrics = calculate_metrics(test_prices.values, lstm_forecast)
    
    print(f"\nLSTM Metrics:")
    print(f"RMSE: ${lstm_metrics['RMSE']:.4f}")
    print(f"MAE:  ${lstm_metrics['MAE']:.4f}")
    print(f"MAPE: {lstm_metrics['MAPE']:.2f}%")

## 7. Model Comparison Summary

In [ ]:
# Compare all models
print("=" * 60)
print("FORECAST MODEL COMPARISON")
print("=" * 60)

comparison_data = {
    'Model': ['ARIMA', 'SARIMA'],
    'Order': [f'{SELECTED_ORDER}', f'{SARIMA_ORDER}x{SEASONAL_ORDER}'],
    'RMSE': [metrics['RMSE'], sarima_metrics['RMSE'] if sarima_forecast is not None else np.nan],
    'MAE': [metrics['MAE'], sarima_metrics['MAE'] if sarima_forecast is not None else np.nan],
    'MAPE (%)': [metrics['MAPE'], sarima_metrics['MAPE'] if sarima_forecast is not None else np.nan]
}

if LSTM_AVAILABLE:
    comparison_data['Model'].append('LSTM')
    comparison_data['Order'].append(f'Window={WINDOW_SIZE}')
    comparison_data['RMSE'].append(lstm_metrics['RMSE'])
    comparison_data['MAE'].append(lstm_metrics['MAE'])
    comparison_data['MAPE (%)'].append(lstm_metrics['MAPE'])

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df.round(4))

# Save comparison
comparison_df.to_csv('../data/processed/model_comparison.csv', index=False)
print("\nModel comparison saved to data/processed/model_comparison.csv")

In [ ]:
# Final visualization: All forecasts together
fig, ax = plt.subplots(figsize=(14, 6))

# Actual
test_prices.plot(ax=ax, label='Actual', color='black', linewidth=2.5)

# ARIMA
forecast.plot(ax=ax, label='ARIMA', color='red', linewidth=1.5, linestyle='--')

# SARIMA
if sarima_forecast is not None:
    sarima_forecast.plot(ax=ax, label='SARIMA', color='blue', linewidth=1.5, linestyle='-.')

# LSTM
if LSTM_AVAILABLE:
    lstm_series = pd.Series(lstm_forecast, index=test_prices.index)
    lstm_series.plot(ax=ax, label='LSTM', color='green', linewidth=1.5, linestyle=':')

ax.set_title(f'{TICKER} Forecast Model Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Close Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/all_forecasts_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/all_forecasts_comparison.png")

## Summary

### Key Findings:

1. **Data Split**: Chronological 80/20 train/test split preserves temporal order

2. **ARIMA Model**: 
   - Selected order based on AIC minimization
   - Parameters (p, d, q) derived from ACF/PACF analysis
   - Provides reasonable baseline forecasts

3. **SARIMA Model**:
   - Adds seasonal component (weekly seasonality)
   - May improve forecast accuracy

4. **LSTM Model**:
   - Deep learning approach with 60-day lookback window
   - Captures non-linear patterns
   - Requires more data and tuning

### Next Steps:
- Apply to TSLA and BND assets
- Implement portfolio optimization
- Create ensemble forecasting approach